In [2]:
import pandas as pd

registrations = pd.read_csv("Registrations.csv")
grades = pd.read_csv("StudentGradesNew.csv")
subjects = pd.read_csv("SubjectInfo.csv")

# Ensure integer types where necessary
registrations["sub_id"] = registrations["sub_id"].astype(int)
registrations["RegEventId"] = registrations["RegEventId"].astype(int)
grades["id"] = grades["id"].astype(int)
grades["RegEventId"] = grades["RegEventId"].astype(int)
subjects["SubId"] = subjects["SubId"].astype(int)
subjects["RegEventId"] = subjects["RegEventId"].astype(int)

print("CSV files loaded successfully.")


CSV files loaded successfully.


In [3]:
registrations['student_id'].nunique()

9180

In [ ]:
import pandas as pd
df=pd.read_csv("SGPA_CGPA.csv")
len(df)


7173

In [17]:
student_id_input = int(input("Enter student_id: "))

student_regs = registrations[registrations["student_id"] == student_id_input]
print("\nStep 1 - Registrations for student:")
display(student_regs)



Step 1 - Registrations for student:


,id,Mode,RegEventId,student_id,sub_id
0,1,1,2,1,1
1,2,1,2,1,2
2,3,1,2,1,3
3,4,1,2,1,4
4,5,1,2,1,5
5,6,1,2,1,6
6,7,1,2,1,7
7,8,1,2,1,8
8,9,1,2,1,9


In [25]:
# Rename _merge if exists
if "_merge" in registrations.columns:
    registrations = registrations.rename(columns={"_merge": "_merge_reg"})

merged_subjects = student_regs.merge(
    subjects, left_on="sub_id", right_on="SubId", how="left", indicator=True
)

print("\nStep 2 - After merging with subjects:")
display(merged_subjects[["sub_id", "SubId", "SubName", "Credits", "_merge"]])

# Optional: see missing subjects
missing_subjects = merged_subjects[merged_subjects["_merge"] == "left_only"]
if not missing_subjects.empty:
    print("Subjects not found in subjects table:", missing_subjects["sub_id"].tolist())



Step 2 - After merging with subjects:


,sub_id,SubId,SubName,Credits,_merge
0,1,1,Environmental Studies,3,both
1,2,2,Problem Solving and Computer Programming,4,both
2,3,3,Problem Solving and Computer Programming Lab,2,both
3,4,4,EAA: Games and Sports - I,0,both
4,5,5,Basic Electronics Engineering,3,both
5,6,6,English for Communication,4,both
6,7,7,Mathematics - I,4,both
7,8,8,Physics,4,both
8,9,9,Physics Lab,2,both


In [26]:
# Use different indicator name to avoid conflict
merged_grades = merged_subjects.merge(
    grades, left_on="RegEventId_x", right_on="RegEventId", how="left", indicator="_merge_grades"
)

print("\nStep 3 - After merging with grades:")
display(merged_grades[["sub_id", "SubName", "AttGrade", "NGrade", "_merge_grades"]])

# Optional: see subjects with no grades yet
missing_grades = merged_grades[merged_grades["_merge_grades"] == "left_only"]
if not missing_grades.empty:
    print("Subjects with no grades yet:", missing_grades["sub_id"].tolist())



Step 3 - After merging with grades:


,sub_id,SubName,AttGrade,NGrade,_merge_grades
0,1,Environmental Studies,P,D,both
1,1,Environmental Studies,P,EX,both
2,1,Environmental Studies,P,D,both
3,1,Environmental Studies,P,EX,both
4,1,Environmental Studies,P,B,both
...,...,...,...,...,...
16438,9,Physics Lab,P,D,both
16439,9,Physics Lab,P,A,both
16440,9,Physics Lab,P,D,both
16441,9,Physics Lab,P,D,both


In [27]:
grade_points = {'O': 10, 'A+': 9, 'A': 8, 'B+': 7, 'B': 6, 'C': 5, 'P': 4, 'F': 0, 'Ab': 0}

merged_grades["NGrade"] = merged_grades["NGrade"].fillna(
    merged_grades["AttGrade"].map(grade_points)
)
merged_grades["TotalPoints"] = merged_grades["Credits"] * merged_grades["NGrade"]

print("\nStep 4 - With TotalPoints:")
display(merged_grades[["sub_id", "SubName", "Credits", "NGrade", "TotalPoints"]])



Step 4 - With TotalPoints:


,sub_id,SubName,Credits,NGrade,TotalPoints
0,1,Environmental Studies,3,D,DDD
1,1,Environmental Studies,3,EX,EXEXEX
2,1,Environmental Studies,3,D,DDD
3,1,Environmental Studies,3,EX,EXEXEX
4,1,Environmental Studies,3,B,BBB
...,...,...,...,...,...
16438,9,Physics Lab,2,D,DD
16439,9,Physics Lab,2,A,AA
16440,9,Physics Lab,2,D,DD
16441,9,Physics Lab,2,D,DD


In [28]:
grade_points = {'O': 10, 'A+': 9, 'A': 8, 'B+': 7, 'B': 6, 'C': 5, 'P': 4, 'F': 0, 'Ab': 0}

merged_grades["NGrade"] = merged_grades["NGrade"].fillna(
    merged_grades["AttGrade"].map(grade_points)
)
merged_grades["TotalPoints"] = merged_grades["Credits"] * merged_grades["NGrade"]

print("\nStep 4 - With TotalPoints:")
display(merged_grades[["sub_id", "SubName", "Credits", "NGrade", "TotalPoints"]])



Step 4 - With TotalPoints:


,sub_id,SubName,Credits,NGrade,TotalPoints
0,1,Environmental Studies,3,D,DDD
1,1,Environmental Studies,3,EX,EXEXEX
2,1,Environmental Studies,3,D,DDD
3,1,Environmental Studies,3,EX,EXEXEX
4,1,Environmental Studies,3,B,BBB
...,...,...,...,...,...
16438,9,Physics Lab,2,D,DD
16439,9,Physics Lab,2,A,AA
16440,9,Physics Lab,2,D,DD
16441,9,Physics Lab,2,D,DD


In [30]:
# Ensure numeric
merged_grades["Credits"] = pd.to_numeric(merged_grades["Credits"], errors="coerce")
merged_grades["TotalPoints"] = pd.to_numeric(merged_grades["TotalPoints"], errors="coerce")

# Compute totals safely
semester_credits = merged_grades["Credits"].sum(skipna=True)
total_points = merged_grades["TotalPoints"].sum(skipna=True)

sgpa = total_points / semester_credits if semester_credits else 0

result_df = pd.DataFrame({
    "RegEventId": [merged_grades["RegEventId_x"].iloc[0] if len(merged_grades) > 0 else None],
    "semester_credits": [semester_credits],
    "SGPA": [round(sgpa, 2)],
    "CGPA_to_date": [round(sgpa, 2)]  # For demo, CGPA = SGPA
})

print("\nFinal SGPA/CGPA table:")
display(result_df)



Final SGPA/CGPA table:


,RegEventId,semester_credits,SGPA,CGPA_to_date
0,2,47502,0.0,0.0
